# Fake News Detection using Natural Language Processing (NLP) & Machine Learning
**Final-Year College NLP Project & Viva Defense Notebook**

---

### Project Overview
Fake news and digital misinformation pose significant challenges to public discourse, elections, and decision-making. This project implements an end-to-end **Natural Language Processing (NLP)** and **Machine Learning (ML)** pipeline to automatically classify news headlines/articles as **REAL** or **FAKE** with confidence scores.

### Key Technical Pipeline
1. **Dataset Ingestion & Exploration**: Structured loading with automatic synthetic sample dataset generation fallback.
2. **Exploratory Data Analysis (EDA)**: Class distribution, text length distributions, and word clouds / frequency analysis.
3. **NLP Preprocessing**: Lowercasing, URL/html removal, tokenization, stopword removal, and **WordNet Lemmatization**.
4. **Feature Extraction**: **TF-IDF Vectorization** with Unigrams and Bigrams (`ngram_range=(1, 2)`).
5. **Multi-Model Training**: Logistic Regression, Multinomial Naive Bayes, and Calibrated Linear SVM.
6. **Model Evaluation & Comparison**: Confusion Matrices, ROC-AUC Curves, Precision, Recall, and F1-Score.
7. **Error Analysis & Interpretability**: Inspecting top feature coefficients and misclassified samples.
8. **Custom Inference Pipeline**: `predict_news()` function with probability-based confidence scores.
9. **Model Persistence**: Serialization of vectorizer and best classifier using `joblib`.
10. **Viva Preparation Guide**: Embedded answers to common NLP and ML viva questions.


## 1. Environment & Library Initialization
In this step, we import core numerical, visualization, NLP, and machine learning libraries. We also download necessary NLTK corpora (`stopwords`, `punkt`, `wordnet`, `omw-1.4`) automatically.


In [ ]:
import os
import re
import string
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional visualization imports with graceful fallbacks
try:
    import seaborn as sns
    HAS_SEABORN = True
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    sns.set_palette('deep')
except ImportError:
    HAS_SEABORN = False
    plt.style.use('ggplot')

try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False

# NLTK imports
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Scikit-Learn imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

# Download required NLTK resources
nltk_resources = ['stopwords', 'punkt', 'wordnet', 'omw-1.4']
for resource in nltk_resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception as e:
        print(f"Note downloading {resource}: {e}")

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("✓ Environment and libraries initialized successfully!")
print(f"Visualization Libraries Installed: Seaborn={HAS_SEABORN}, WordCloud={HAS_WORDCLOUD}")


## 2. Dataset Ingestion & Pre-Exploration
We build a smart dataset loader function. If a standard local dataset (`Fake.csv`/`True.csv`, `train.csv`, `fake_news.csv`) is present, it will automatically load and structure it. Otherwise, it generates a balanced, realistic synthetic dataset containing distinct stylistic signals for **REAL** (formal tone, official institutions, economic metrics) and **FAKE** (sensationalism, clickbait, conspiratorial terms).


In [ ]:
def load_or_create_dataset():
    possible_files = ['fake_news.csv', 'train.csv', 'Fake.csv', 'True.csv', 'fake_or_real_news.csv']
    
    for fname in possible_files:
        if os.path.exists(fname):
            print(f"Found existing local dataset file: '{fname}'")
            try:
                df = pd.read_csv(fname)
                df.columns = [c.lower().strip() for c in df.columns]
                if 'label' in df.columns and 'text' in df.columns:
                    return df[['text', 'label']].dropna()
                elif 'title' in df.columns and 'text' in df.columns and 'label' in df.columns:
                    df['combined'] = df['title'].fillna('') + " " + df['text'].fillna('')
                    return df[['combined', 'label']].rename(columns={'combined': 'text'}).dropna()
            except Exception as err:
                print(f"Could not read {fname}: {err}. Falling back...")

    print("No local CSV dataset detected. Creating synthetic sample dataset for full execution...")
    
    fake_samples = [
        "BREAKING: Secret government document leaked showing alien technology covered up by NASA for decades!",
        "UNBELIEVABLE Miracle cure for diabetes hidden by big pharma executives to keep profits soaring!",
        "SHOCKING Scandal! Prominent politician caught secretly transferring millions to offshore tax havens overnight!",
        "CONFIRMED: Scientists discover drinking lemon water with salt cures all viral infections instantly!",
        "EXPOSED: Celebrity reveals secret scheme used by elites to manipulate global stock markets!",
        "YOU WON'T BELIEVE WHAT HAPPENED! Local man discovers unlimited free energy using simple household magnets!",
        "ALERT: Hidden camera footage reveals secret meeting planning total internet shutdown next week!",
        "BOMBSHELL report: Whistleblower proves election voting machines were hacked by offshore syndicate!",
        "URGENT: Government planning secret law to ban cash transactions starting next month!",
        "REVEALED: Ancient map found in cave reveals hidden city of gold beneath the pyramids!",
        "STUNNING: Famous actor admits entire Hollywood movie industry is controlled by secret society!",
        "MUST SEE: Mysterious glowing object spotted over capital city, authorities refuse to comment!",
        "SENSATIONAL: New smartphone update secretly records private conversations and sends to advertisers!",
        "MIRACLE remedy found in rain forest eliminates gray hair and restores youth in 7 days!",
        "HORRIFYING truth behind common food additives exposed by former food inspector!",
        "BREAKING NEWS: Secret portal discovered in Antarctica scientists claim leads to parallel universe!",
        "UNBELIEVABLE: Man wins lottery five times in a row using ancient mathematical trick!",
        "SHOCKING: Leaked email reveals major tech company tracking every user movement without consent!",
        "EXCLUSIVE: Insider leaks details of secret underground bunker built for billionaire executives!",
        "URGENT WARNING: Drinking tap water leads to memory loss according to rogue researcher!",
        "BREAKING: Billionaire investor predicts total collapse of financial system by end of the week!",
        "SCANDAL: Top university professor exposed for fabricating climate change data for funding!",
        "SHOCKING DISCOVERY: Eating midnight snacks causes immediate heart damage according to viral blog!",
        "CONFIRMED: Secret technology allows weather manipulation to cause artificial storms!",
        "EXPOSED: Major bank accidentally credited customer accounts with millions and refuses to refund!",
    ] * 20

    real_samples = [
        "Federal Reserve announces 25 basis point interest rate adjustment following quarterly economic report.",
        "World Health Organization releases updated global public health guidelines on seasonal influenza prevention.",
        "NASA successfully launches new orbital satellite to study ocean currents and atmospheric moisture levels.",
        "Department of Labor reports unemployment claims dropped to lowest level in six months.",
        "United Nations climate summit concludes with international agreement on reducing carbon emissions by 2030.",
        "European Central Bank maintains key benchmark rates steady amidst stabilizing inflation metrics.",
        "Research team at Oxford University publishes peer-reviewed study on renewable solar cell efficiency.",
        "Supreme Court hears oral arguments regarding federal regulatory authority in inter-state commerce.",
        "Global supply chain disruptions show signs of easing as port congestion decreases nationwide.",
        "Ministry of Health confirms nationwide vaccination campaign lowered hospital admissions significantly.",
        "National Science Foundation awards grant for advanced quantum computing research project.",
        "Agricultural department forecasts record wheat harvest following favorable spring rainfall conditions.",
        "Tech consortium establishes new open-source cybersecurity standards for enterprise cloud systems.",
        "Transport security agency implements upgraded biometric screening protocols at major international airports.",
        "Urban planning committee approves expansion plan for municipal light rail transit system.",
        "Geological survey detects minor earthquake off Pacific coast; no tsunami warning issued.",
        "Bureau of Economic Analysis updates second quarter GDP growth estimate to 2.4 percent annualized.",
        "International Monetary Fund releases annual outlook report highlighting emerging market resilience.",
        "State governor signs bipartisan infrastructure bill allocating funds for bridge and highway repairs.",
        "Environmental Protection Agency issues updated air quality standards for industrial manufacturing facilities.",
        "Telecommunications regulatory agency mandates clearer pricing disclosures for broadband providers.",
        "Energy Information Administration reports unexpected build in commercial crude oil inventories.",
        "Department of Education announces expansion of STEM apprenticeship programs in rural community colleges.",
        "World Trade Organization resolves long-standing trade dispute between member nations regarding agricultural tariffs.",
        "National Weather Service issues winter storm advisory for northern plains regions ahead of cold front.",
    ] * 20

    fake_df = pd.DataFrame({'text': fake_samples, 'label': 1})
    real_df = pd.DataFrame({'text': real_samples, 'label': 0})
    
    df = pd.concat([fake_df, real_df], ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
    return df

df = load_or_create_dataset()

# Format numeric label to string name
if df['label'].dtype in [np.int64, np.int32, int]:
    df['label_name'] = df['label'].map({0: 'REAL', 1: 'FAKE'})
else:
    df['label_name'] = df['label'].astype(str).str.upper()
    df['label'] = df['label_name'].map({'REAL': 0, 'FAKE': 1})

print(f"Dataset Loaded Successfully! Total Samples: {len(df)}")
print("\nFirst 5 Dataset Rows:")
display(df.head())


## 3. Exploratory Data Analysis (EDA)
Exploratory Data Analysis allows us to understand the data distribution, check class balance, analyze text length variations, and identify frequent terms in **REAL** vs **FAKE** news.


In [ ]:
# 1. Class Distribution Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['label_name'].value_counts()
if HAS_SEABORN:
    sns.countplot(data=df, x='label_name', palette=['#2ecc71', '#e74c3c'], ax=axes[0])
else:
    axes[0].bar(counts.index, counts.values, color=['#2ecc71', '#e74c3c'])

axes[0].set_title('Class Distribution (Real vs Fake News)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Count')

for p in (axes[0].patches if HAS_SEABORN else axes[0].containers[0]):
    height = p.get_height()
    axes[0].annotate(f'{int(height)}', (p.get_x() + p.get_width() / 2., height / 2),
                     ha='center', va='center', fontsize=12, color='white', fontweight='bold')

counts.plot.pie(
    autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], ax=axes[1],
    startangle=90, explode=(0.03, 0.03), textprops={'fontsize': 12, 'weight': 'bold'}
)
axes[1].set_title('Dataset Class Balance', fontsize=13, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# 2. Text Length Feature Engineering
df['char_count'] = df['text'].astype(str).apply(len)
df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if HAS_SEABORN:
    sns.histplot(data=df, x='word_count', hue='label_name', kde=True, element="step",
                 palette={'REAL': '#2ecc71', 'FAKE': '#e74c3c'}, ax=axes[0])
    sns.boxplot(data=df, x='label_name', y='word_count', palette=['#2ecc71', '#e74c3c'], ax=axes[1])
else:
    axes[0].hist(df[df['label']==0]['word_count'], alpha=0.6, label='REAL', color='#2ecc71')
    axes[0].hist(df[df['label']==1]['word_count'], alpha=0.6, label='FAKE', color='#e74c3c')
    axes[0].legend()
    axes[1].boxplot([df[df['label']==0]['word_count'], df[df['label']==1]['word_count']], labels=['REAL', 'FAKE'])

axes[0].set_title('Word Count Distribution by Class', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Word Count per Article')
axes[0].set_ylabel('Frequency')

axes[1].set_title('Word Count Boxplot Comparison', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

print("Summary Text Statistics:")
display(df.groupby('label_name')[['char_count', 'word_count']].describe().T)


In [ ]:
# Word Cloud / Word Frequency Visualizations
real_text = " ".join(df[df['label'] == 0]['text'])
fake_text = " ".join(df[df['label'] == 1]['text'])

if HAS_WORDCLOUD:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    wc_real = WordCloud(width=800, height=400, background_color='white', colormap='Greens', max_words=100).generate(real_text)
    axes[0].imshow(wc_real, interpolation='bilinear')
    axes[0].set_title('Word Cloud - REAL News Articles', fontsize=14, fontweight='bold', color='green')
    axes[0].axis('off')

    wc_fake = WordCloud(width=800, height=400, background_color='black', colormap='Reds', max_words=100).generate(fake_text)
    axes[1].imshow(wc_fake, interpolation='bilinear')
    axes[1].set_title('Word Cloud - FAKE News Articles', fontsize=14, fontweight='bold', color='red')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    from collections import Counter
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    real_words = [w.lower() for w in real_text.split() if len(w) > 4][:100]
    fake_words = [w.lower() for w in fake_text.split() if len(w) > 4][:100]
    
    top_real = pd.DataFrame(Counter(real_words).most_common(10), columns=['word', 'count'])
    top_fake = pd.DataFrame(Counter(fake_words).most_common(10), columns=['word', 'count'])
    
    axes[0].barh(top_real['word'], top_real['count'], color='#2ecc71')
    axes[0].set_title('Top Words in REAL News')
    axes[0].invert_yaxis()
    
    axes[1].barh(top_fake['word'], top_fake['count'], color='#e74c3c')
    axes[1].set_title('Top Words in FAKE News')
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()


## 4. NLP Text Preprocessing Pipeline
Text preprocessing converts unstructured, noisy raw text into clean, standardized tokens suitable for vectorization.

### Preprocessing Steps:
1. **Lowercasing**: Normalizes casing so `"BREAKING"` and `"breaking"` are mapped to the same feature.
2. **Noise Removal**: Strips URLs (`http...`), HTML tags (`<...>`), email addresses, special symbols, and digits.
3. **Tokenization**: Splits text strings into individual word tokens using NLTK `word_tokenize`.
4. **Stopword Removal**: Filters out uninformative grammatical words (`is`, `the`, `and`, `at`).
5. **Lemmatization**: Uses NLTK `WordNetLemmatizer` to reduce inflected words to their canonical dictionary lemma (*e.g.*, *announces* $ightarrow$ *announce*, *hacked* $ightarrow$ *hack*).

> **Lemmatization vs. Stemming (Viva Note)**: Stemming uses heuristic rules to chop off word ends (often producing non-words like *connecti*), whereas Lemmatization uses morphological lookup with vocabulary to return true dictionary root forms.


In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove URLs, HTML tags, emails
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    
    # 3. Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 4. Tokenization
    try:
        tokens = word_tokenize(text)
    except Exception:
        tokens = text.split()
    
    # 5. Stopword removal & Lemmatization
    cleaned_tokens = [
        lemmatizer.lemmatize(word) for word in tokens 
        if word not in stop_words and len(word) > 2
    ]
    
    return " ".join(cleaned_tokens)

print("Executing Preprocessing Pipeline across entire dataset...")
df['clean_text'] = df['text'].apply(preprocess_text)

print("\n--- Raw vs Cleaned Text Comparison ---")
for idx, row in df.head(3).iterrows():
    print(f"\n[Sample {idx+1}] Label: {row['label_name']}")
    print(f"RAW   : {row['text'][:110]}...")
    print(f"CLEAN : {row['clean_text'][:110]}...")


## 5. Train-Test Split & TF-IDF Feature Extraction

### Data Splitting
We apply a **80/20 Stratified Train-Test Split** to ensure that both training and evaluation subsets preserve identical class proportions.

### TF-IDF Vectorization
**TF-IDF** (Term Frequency - Inverse Document Frequency) measures how important a word is to a document relative to a corpus:

$$\text{TF}(t, d) = \frac{\text{Count of term } t \text{ in document } d}{\text{Total terms in document } d}$$

$$\text{IDF}(t) = \log\left(\frac{N}{1 + \text{DF}(t)}\right)$$

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

We configure `ngram_range=(1, 2)` to extract both single words (**Unigrams**) and two-word combinations (**Bigrams**), capturing contextual phrases such as *"secret document"* or *"climate change"*.

> **Data Leakage Prevention (Viva Note)**: We fit `TfidfVectorizer` **only on the training set** (`X_train`) and transform `X_test` using the fitted vocabulary. Fitting on the entire dataset prior to splitting leaks test distribution information.


In [ ]:
# 1. Stratified Train-Test Split (80% Train, 20% Test)
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set size : {X_train.shape[0]} samples")
print(f"Test set size     : {X_test.shape[0]} samples")

# 2. Fit TF-IDF Vectorizer on X_train ONLY
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000,
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"\nTF-IDF Train Matrix Shape : {X_train_tfidf.shape}")
print(f"TF-IDF Test Matrix Shape  : {X_test_tfidf.shape}")
print(f"Extracted Vocabulary Size : {len(tfidf_vectorizer.vocabulary_)} n-gram features")


## 6. Machine Learning Model Training & Calibration
We train and evaluate three diverse machine learning classifiers:

1. **Logistic Regression**: A linear model using the sigmoid function to output continuous probability estimates $P(y=1|x) = \frac{1}{1 + e^{-w^T x}}$.
2. **Multinomial Naive Bayes**: A probabilistic algorithm applying Bayes' Theorem under the conditional independence assumption $P(y|x) \propto P(y) \prod P(x_i|y)$. Ideal for discrete text counts.
3. **Linear SVM (Support Vector Machine)**: Finds the maximum-margin decision boundary. Wrapped in `CalibratedClassifierCV` (Platt Scaling) to enable reliable probability predictions for confidence scoring.

### Performance Metrics Evaluated:
- **Accuracy**: Overall proportion of correct predictions.
- **Precision**: $\frac{TP}{TP + FP}$ (Minimizes False Positives: calling Real news Fake).
- **Recall**: $\frac{TP}{TP + FN}$ (Minimizes False Negatives: missing actual Fake news).
- **F1-Score**: Harmonic mean of Precision and Recall: $2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$.
- **ROC-AUC**: Area under the Receiver Operating Characteristic curve.


In [ ]:
# Define classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Linear SVM': CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42), method='sigmoid')
}

results = []
trained_models = {}
y_probs = {}
y_preds = {}

print("Training Machine Learning Classifiers...")
for name, model in classifiers.items():
    model.fit(X_train_tfidf, y_train)
    
    pred = model.predict(X_test_tfidf)
    prob = model.predict_proba(X_test_tfidf)[:, 1]
    
    trained_models[name] = model
    y_preds[name] = pred
    y_probs[name] = prob
    
    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred)
    rec = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    auc = roc_auc_score(y_test, prob)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'ROC-AUC': auc
    })

results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False).reset_index(drop=True)
print("\n--- Model Performance Comparison Summary ---")
display(results_df)


## 7. Model Evaluation, Visualization & Best Model Selection
Visual evaluation helps us analyze error distributions across classes and compare classification thresholds.


In [ ]:
# 1. Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, pred) in enumerate(y_preds.items()):
    cm = confusion_matrix(y_test, pred)
    if HAS_SEABORN:
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                    xticklabels=['REAL', 'FAKE'], yticklabels=['REAL', 'FAKE'],
                    cbar=False, annot_kws={'size': 14, 'weight': 'bold'})
    else:
        axes[idx].imshow(cm, cmap='Blues')
        for i in range(2):
            for j in range(2):
                axes[idx].text(j, i, str(cm[i, j]), ha='center', va='center', color='red', fontsize=14, fontweight='bold')
        axes[idx].set_xticks([0, 1])
        axes[idx].set_yticks([0, 1])
        axes[idx].set_xticklabels(['REAL', 'FAKE'])
        axes[idx].set_yticklabels(['REAL', 'FAKE'])
        
    axes[idx].set_title(f'Confusion Matrix\n{name}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('Actual Label')

plt.tight_layout()
plt.show()

# 2. Combined ROC Curves Plot
plt.figure(figsize=(9, 6))

for name, prob in y_probs.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Baseline (AUC = 0.5000)')
plt.title('Receiver Operating Characteristic (ROC) Curves', fontsize=14, fontweight='bold')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True)
plt.show()

# 3. Performance Metrics Bar Chart
results_melted = results_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(12, 6))
if HAS_SEABORN:
    sns.barplot(data=results_melted, x='Metric', y='Score', hue='Model', palette='viridis')
else:
    metrics = results_df.columns[1:]
    x_coords = np.arange(len(metrics))
    width = 0.25
    for i, m_name in enumerate(results_df['Model']):
        plt.bar(x_coords + i*width, results_df.iloc[i][metrics], width=width, label=m_name)
    plt.xticks(x_coords + width, metrics)

plt.title('Model Metrics Comparison', fontsize=14, fontweight='bold')
plt.ylim(0.0, 1.05)
plt.legend(loc='lower right')
plt.grid(axis='y')
plt.show()

# Select Best Model
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
print(f"★ SELECTED BEST MODEL: >>> {best_model_name} <<< (F1-Score: {results_df.iloc[0]['F1-Score']:.4f}, ROC-AUC: {results_df.iloc[0]['ROC-AUC']:.4f})")


## 8. Error Analysis & Feature Importance Inspection
Understanding *why* a model makes a decision is essential for transparency in NLP. We inspect:
1. **Top TF-IDF Feature Coefficients**: The words/phrases most strongly driving predictions toward **FAKE** vs **REAL**.
2. **Misclassification Inspection**: Examining samples where actual labels differed from predicted labels.


In [ ]:
# Feature Importance Extraction
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

if hasattr(best_model, 'coef_'):
    coefs = best_model.coef_[0]
elif hasattr(best_model, 'calibrated_classifiers_'):
    coefs = np.mean([clf.base_estimator.coef_[0] for clf in best_model.calibrated_classifiers_], axis=0)
else:
    coefs = trained_models['Logistic Regression'].coef_[0]

top_fake_idx = np.argsort(coefs)[-15:]
top_real_idx = np.argsort(coefs)[:15]

top_fake_features = pd.DataFrame({'Word/Phrase': feature_names[top_fake_idx], 'Weight': coefs[top_fake_idx]})
top_real_features = pd.DataFrame({'Word/Phrase': feature_names[top_real_idx], 'Weight': coefs[top_real_idx]})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if HAS_SEABORN:
    sns.barplot(data=top_fake_features.sort_values(by='Weight', ascending=True),
                y='Word/Phrase', x='Weight', palette='Reds_r', ax=axes[0])
    sns.barplot(data=top_real_features.sort_values(by='Weight', ascending=False),
                y='Word/Phrase', x='Weight', palette='Greens', ax=axes[1])
else:
    df_f = top_fake_features.sort_values(by='Weight', ascending=True)
    df_r = top_real_features.sort_values(by='Weight', ascending=False)
    axes[0].barh(df_f['Word/Phrase'], df_f['Weight'], color='red')
    axes[1].barh(df_r['Word/Phrase'], df_r['Weight'], color='green')

axes[0].set_title('Top 15 TF-IDF Features Indicative of FAKE News (+ Weight)', fontsize=12, fontweight='bold')
axes[1].set_title('Top 15 TF-IDF Features Indicative of REAL News (- Weight)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Misclassification Deep-Dive
test_indices = y_test.index
best_preds = y_preds[best_model_name]
best_probs = y_probs[best_model_name]

error_df = pd.DataFrame({
    'Original_Text': df.loc[test_indices, 'text'].values,
    'Actual_Label': y_test.map({0: 'REAL', 1: 'FAKE'}).values,
    'Predicted_Label': pd.Series(best_preds).map({0: 'REAL', 1: 'FAKE'}).values,
    'Fake_Probability': best_probs
})

misclassified = error_df[error_df['Actual_Label'] != error_df['Predicted_Label']]

print(f"\nTotal Misclassified Test Samples: {len(misclassified)} out of {len(y_test)}")
if len(misclassified) > 0:
    display(misclassified[['Original_Text', 'Actual_Label', 'Predicted_Label', 'Fake_Probability']].head(5))
else:
    print("Zero misclassifications on this test split!")


## 9. Custom News Prediction & Confidence Pipeline
We encapsulate the entire preprocessing, vectorization, and inference flow into a user-friendly function `predict_news()`. It accepts raw news text and outputs:
- Classification label (**REAL** or **FAKE**)
- Model Confidence Score (%)
- Probabilities $P(\text{REAL})$ and $P(\text{FAKE})$
- Risk/Trust Assessment report.


In [ ]:
def predict_news(news_text, model=best_model, vectorizer=tfidf_vectorizer):
    cleaned_input = preprocess_text(news_text)
    
    if not cleaned_input.strip():
        return {
            'Text Snippet': news_text[:80],
            'Prediction': 'UNKNOWN',
            'Confidence': '0.00%',
            'Prob(REAL)': '0.0000',
            'Prob(FAKE)': '0.0000',
            'Assessment': 'INVALID INPUT: Text contains no valid vocabulary tokens.'
        }
    
    vec_input = vectorizer.transform([cleaned_input])
    probabilities = model.predict_proba(vec_input)[0]
    prob_real, prob_fake = probabilities[0], probabilities[1]
    
    prediction = 'FAKE' if prob_fake >= 0.5 else 'REAL'
    confidence = max(prob_fake, prob_real) * 100
    
    if prediction == 'FAKE':
        if confidence > 85:
            risk = 'HIGH RISK: Highly sensationalist / unverified clickbait tone.'
        elif confidence > 65:
            risk = 'MODERATE RISK: Contains sensational phrases.'
        else:
            risk = 'BORDERLINE FAKE: Marginally classified as fake news.'
    else:
        if confidence > 85:
            risk = 'HIGH TRUST: Formatted like official, verifiable news reporting.'
        else:
            risk = 'MODERATE TRUST: Standard objective news phrasing.'
            
    return {
        'Text Snippet': news_text[:90] + ('...' if len(news_text) > 90 else ''),
        'Prediction': prediction,
        'Confidence': f"{confidence:.2f}%",
        'Prob(REAL)': f"{prob_real:.4f}",
        'Prob(FAKE)': f"{prob_fake:.4f}",
        'Assessment': risk
    }

# Test Custom Real-World News Samples
sample_test_headlines = [
    "SHOCKING: Scientists secretly clone dinosaur in underground laboratory and hide it from the public!",
    "The Federal Reserve announced an adjustment to interest rates following the quarterly inflation summary.",
    "YOU WON'T BELIEVE THIS MIRACLE CURE FOR ILLNESSES! Doctors are furious!",
    "NASA launched a new weather monitoring satellite into low Earth orbit on Tuesday morning."
]

print("="*85)
print("TESTING CUSTOM INFERENCE PIPELINE WITH CONFIDENCE SCORES")
print("="*85)

test_results = [predict_news(text) for text in sample_test_headlines]
display(pd.DataFrame(test_results))


## 10. Model Saving & Serialization
To deploy the trained model in production (e.g. via Flask, FastAPI, or Streamlit), we serialize the trained vectorizer and model weights using `joblib`.


In [ ]:
# Save model and vectorizer
os.makedirs('saved_models', exist_ok=True)

model_path = os.path.join('saved_models', 'fake_news_best_model.pkl')
vectorizer_path = os.path.join('saved_models', 'tfidf_vectorizer.pkl')

joblib.dump(best_model, model_path)
joblib.dump(tfidf_vectorizer, vectorizer_path)

print(f"✓ Model successfully saved to       : {model_path}")
print(f"✓ Vectorizer successfully saved to  : {vectorizer_path}")

# Load and verify inference
loaded_model = joblib.load(model_path)
loaded_vectorizer = joblib.load(vectorizer_path)

verify_text = "BOMBSHELL: Secret document proves aliens landing next month!"
verification_res = predict_news(verify_text, model=loaded_model, vectorizer=loaded_vectorizer)

print("\n--- Saved Model Verification Test ---")
print(f"Input Text : {verify_text}")
print(f"Prediction : {verification_res['Prediction']} ({verification_res['Confidence']})")
print("Status     : SUCCESS! Saved model restored and working perfectly.")


## 11. Conclusion & Viva Defense Cheat Sheet

### Summary of Achievements
- Successfully built an original end-to-end Fake News Detection system using NLP and Machine Learning.
- Extracted unigram and bigram TF-IDF features (`ngram_range=(1, 2)`), capturing key phrase patterns.
- Evaluated **Logistic Regression**, **Multinomial Naive Bayes**, and **Linear SVM**, selecting the optimal model based on F1-Score and ROC-AUC metrics.
- Developed an interactive `predict_news()` inference pipeline returning probability-based confidence scores.
- Serialized the complete model artifact for real-world deployment.

---

### Viva Defense Cheat Sheet (Key Theoretical Q&A)

#### Q1: Why use TF-IDF instead of simple Count Vectorizer (Bag-of-Words)?
> **Answer**: `CountVectorizer` only measures term frequencies, giving high weight to words that appear repeatedly everywhere. **TF-IDF** multiplies Term Frequency by Inverse Document Frequency, penalizing common corpus words and highlighting distinctive, domain-informative keywords.

#### Q2: What is the purpose of Bigrams `ngram_range=(1, 2)`?
> **Answer**: Single unigrams lose contextual associations (*e.g.*, *"breaking"* and *"news"* separately vs. *"breaking news"* together). Bigrams retain key word pairs (*e.g.*, *"climate change"*, *"secret document"*, *"miracle cure"*), significantly improving classification context.

#### Q3: Why is Lemmatization preferred over Stemming for Fake News Classification?
> **Answer**: Stemming applies crude heuristic rules that cut off word endings, often generating non-words (*e.g.*, *sensat* for *sensational*). Lemmatization uses morphological lookup with WordNet POS tagging to return valid dictionary root words (*lemmas*), maintaining semantic integrity.

#### Q4: How did you prevent Data Leakage during feature extraction?
> **Answer**: We performed the train-test split **before** vectorization, fitting `TfidfVectorizer` strictly on `X_train`. `X_test` was transformed using the training vocabulary only, preventing test information from leaking into model training.

#### Q5: Why wrap Linear SVM in `CalibratedClassifierCV`?
> **Answer**: Standard Linear SVM computes hyper-plane distance margins ($w^T x + b$), which are un-normalized and cannot be interpreted directly as probabilities. `CalibratedClassifierCV` uses **Platt Scaling** (fitting a sigmoid curve over margins) to output normalized probability distributions needed for confidence scoring.

#### Q6: What are potential future enhancements for this project?
> **Answer**:
> 1. Fine-tuning deep contextual Transformer models such as **BERT**, **RoBERTa**, or **DeBERTa**.
> 2. Incorporating metadata features (author domain history, publisher credibility scores, social media engagement patterns).
> 3. Multi-modal detection (combining text analysis with deep learning image forgery detection).
